In [1]:
%%time
%load_ext autoreload
%autoreload 2

import pandas as pd
import wandb
import pandas as pd
from collections import defaultdict

# Table processing
def process_line(means, highlight, highlight_index, highlight_max, ignore_std):
    if highlight:
        if highlight_max:
            tops = set(means.groupby(highlight_index).idxmax())
        else:
            tops = set(means.groupby(highlight_index).idxmin())
    else:
        tops = set()

    def process_line(x):
        if ignore_std:
            if x.name in tops:
                return rf"\textbf{{{x['mean']:0.3f}}}"
            return rf"{x['mean']:0.3f}"
        if x.name in tops:
            return rf"\textbf{{{x['mean']:0.3f} $\pm$ {x['std']:0.3f}}}"
        return rf"{x['mean']:0.3f} $\pm$ {x['std']:0.3f}"

    return process_line


def mean_pm_std(
    data,
    index,
    columns,
    value,
    highlight=True,
    highlight_cols=True,
    highlight_max=True,
    ignore_std=False,
):
    assert len(data) > 0
    groupby = data.groupby([*index, *columns])
    means = groupby.mean()[value].rename("mean")
    stds = groupby.std()[value].rename("std")
    ddf = pd.concat([means, stds], axis=1).T
    highlight_index = columns if highlight_cols else index
    ddf = ddf.apply(
        process_line(means, highlight, highlight_index, highlight_max, ignore_std)
    )
    ddf = ddf.reset_index().pivot(index=index, columns=columns)
    ddf.columns = ddf.columns.droplevel(level=0)
    return ddf

    
def flatten_dict(d, parent_key="", sep="/"):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)


def prepare_data(data):
    flattened_data = [flatten_dict(item) for item in data]
    return pd.DataFrame(flattened_data)



CPU times: user 1.01 s, sys: 136 ms, total: 1.14 s
Wall time: 1.19 s


In [2]:
api = wandb.Api(timeout=30)

# Project is specified by <entity/project-name>
runs = api.runs(
    "openproblems-comp/fast-tbg",
    filters={
        "$and": [
            {
                # "created_at": {"$gt": "2025-01-26T00:00:00"},
                "tags": {"$in": ["jarz_final_iv2"]},
                #'group': {'$in': ['5_vars']},
                # "config.data.n_particles": {"$eq": 22},
                #'config.model': {'$eq': model},
                #'config.lr': {'$lt': 1.01 * lr, '$gt': 0.99 * lr},
            }
        ]
    },
)

summary_list, config_list, name_list, tag_list = [], [], [], []
for run in runs:
    tag_list.append(run.tags)
    # .summary contains the output keys/values for metrics like accuracy.
    #  We call ._json_dict to omit large files
    summary_list.append(run.summary._json_dict)
    # .config contains the hyperparameters.
    #  We remove special values that start with _.
    config_list.append({k: v for k, v in run.config.items() if not k.startswith("_")})
    # .name is the human-readable name of the run.
    name_list.append(run.name)
df_summary = prepare_data(summary_list)
df_config = prepare_data(config_list)
tag_list = [str(t) for t in tag_list]
df = pd.concat(
    [
        pd.DataFrame(name_list, columns=["name"]),
        pd.DataFrame(tag_list, columns=["Tags"]),
        df_summary,
        df_config,
    ],
    axis=1,
)

In [3]:
metrics = [
    "test/samples_walltime",
    "test/jarzynski/samples_walltime",
]

In [4]:
import math


def filterer(x):
    if isinstance(x, float) and not math.isfinite(x):
        return False
    return "table" in list(x)


filtered_df = df[
    # df["model/_target_"].isin(["src.models.flow_matching_module.FlowMatchLitModule"])
    # & df["Tags"].apply(lambda x: "eval" in x)
    df["Tags"].apply(lambda x: True)
    # & ~df["val/cropped_energy_w1"].isna()
    # & df["tags"].apply(filterer)
][
    [
        # "tags",
        "model/_target_",
        "model/net/_target_",
        "data/n_particles",
        *metrics,
    ]
].drop(
    columns=["model/_target_"]
)

# filtered_df.sort_values("data/n_particles")

In [5]:
filtered_df["test/samples_walltime"] = filtered_df["test/samples_walltime"].astype(float) / 60 / 60
filtered_df["test/jarzynski/samples_walltime"] = (
    filtered_df["test/jarzynski/samples_walltime"].astype(float) / 60 / 60
)

filtered_df["test/samples+jarzynski_walltime"] = (
    filtered_df["test/samples_walltime"] + filtered_df["test/jarzynski/samples_walltime"]
)

metrics.append("test/samples+jarzynski_walltime")

In [6]:
renamed_df = filtered_df.replace(
    {
        "src.models.components.tbg.egnn_dynamics_ad2_cat.EGNN_dynamics_AD2_cat": "EQ-CFM",
        "src.models.components.dit.DIT3D": "DiT-CFM",
        "src.models.components.tarflow.TarFlow": "TarFlow",
    }
).rename(
    columns={
        "model/net/_target_": "Model",
        "data/n_particles": "n_particles",
        "test/cropped_energy_w1": "test/fcropped_energy_w1",
    }
)  # so order is correct lol

renamed_df.rename(
    columns={
        "test/samples_walltime": "0test/samples_walltime",
        "test/jarzynski/samples_walltime": "1test/jarzynski/samples_walltime",
        "test/samples+jarzynski_walltime": "2test/samples+jarzynski_walltime",
    },
    inplace=True,
)

metrics[metrics.index("test/samples_walltime")] = "0test/samples_walltime"
metrics[metrics.index("test/jarzynski/samples_walltime")] = "1test/jarzynski/samples_walltime"
metrics[metrics.index("test/samples+jarzynski_walltime")] = "2test/samples+jarzynski_walltime"

In [7]:
renamed_df.groupby(["Model", "n_particles"]).count()

0test/samples_walltime  1test/jarzynski/samples_walltime  \
Model   n_particles                                                             
TarFlow 22                                9                                 9   
        33                                9                                 9   
        42                                9                                 9   

                     2test/samples+jarzynski_walltime  
Model   n_particles                                    
TarFlow 22                                          9  
        33                                          9  
        42                                          9

In [8]:
df_melt = renamed_df.dropna().melt(
    value_vars=metrics, id_vars=["Model", "n_particles"], var_name="Metric"
)

In [9]:
results = mean_pm_std(
    df_melt, index=["Model"], columns=["n_particles", "Metric"], value="value", highlight=False
)
results

n_particles                     22                                   \
Metric      0test/samples_walltime 1test/jarzynski/samples_walltime   
Model                                                                 
TarFlow          0.002 $\pm$ 0.000                0.108 $\pm$ 0.001   

n_particles                                                      33  \
Metric      2test/samples+jarzynski_walltime 0test/samples_walltime   
Model                                                                 
TarFlow                    0.109 $\pm$ 0.001      0.002 $\pm$ 0.000   

n_particles                                                                    \
Metric      1test/jarzynski/samples_walltime 2test/samples+jarzynski_walltime   
Model                                                                           
TarFlow                    0.130 $\pm$ 0.001                0.132 $\pm$ 0.001   

n_particles                     42                                   \
Metric      0test/samples_walltime 1test/jarzynski/samples_walltime   
Model                                                                 
TarFlow          0.003 $\pm$ 0.000                0.235 $\pm$ 0.001   

n_particles                                   
Metric      2test/samples+jarzynski_walltime  
Model                                         
TarFlow                    0.239 $\pm$ 0.001

In [10]:
print(
    results.to_latex(
        float_format="{:.3f}".format,
    )
)

\begin{tabular}{llllllllll}
\toprule
n\_particles & \multicolumn{3}{l}{22} & \multicolumn{3}{l}{33} & \multicolumn{3}{l}{42} \\
Metric & 0test/samples\_walltime & 1test/jarzynski/samples\_walltime & 2test/samples+jarzynski\_walltime & 0test/samples\_walltime & 1test/jarzynski/samples\_walltime & 2test/samples+jarzynski\_walltime & 0test/samples\_walltime & 1test/jarzynski/samples\_walltime & 2test/samples+jarzynski\_walltime \\
Model   &                        &                                  &                                  &                        &                                  &                                  &                        &                                  &                                  \\
\midrule
TarFlow &      0.002 \$\textbackslash pm\$ 0.000 &                0.108 \$\textbackslash pm\$ 0.001 &                0.109 \$\textbackslash pm\$ 0.001 &      0.002 \$\textbackslash pm\$ 0.000 &                0.130 \$\textbackslash pm\$ 0.001 &                0.1